# 04 - Fakturoid Integration

This notebook tests integration with Fakturoid API for submitting invoices.


In [1]:
from src.fakturoid_client import FakturoidClient
from src.config import config
import json


ModuleNotFoundError: No module named 'ai_extractor'

In [ ]:
# Initialize Fakturoid client
fakturoid = FakturoidClient(config)

print("Fakturoid Client initialized")
print(f"Account: {config.fakturoid.account_slug}")
print(f"Base URL: {config.fakturoid.base_url}")


In [ ]:
# Test connection - get account info
try:
    account_info = fakturoid.get_account_info()
    print("✓ Successfully connected to Fakturoid")
    print(f"\nAccount Info:")
    print(json.dumps(account_info, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"✗ Failed to connect to Fakturoid: {e}")


In [ ]:
# List existing subjects (suppliers)
try:
    subjects = fakturoid.list_subjects()
    print(f"Found {len(subjects)} subjects in Fakturoid:")
    for subject in subjects[:5]:  # Show first 5
        print(f"  - {subject.get('name')} (ID: {subject.get('id')})")
    if len(subjects) > 5:
        print(f"  ... and {len(subjects) - 5} more")
except Exception as e:
    print(f"✗ Failed to list subjects: {e}")


In [ ]:
# Test ARES lookup for Czech companies
print("Testing ARES (Czech Business Register) lookup")
print("="*60)

# Test with a real Czech company IČO
test_ico = "24167185"  # Bohemia Falcon Studio
print(f"\nLooking up IČO: {test_ico}")

ares_data = fakturoid.get_company_from_ares(test_ico)

if ares_data:
    print(f"\n✓ Company found in ARES:")
    print(f"  Name: {ares_data.get('name')}")
    print(f"  Street: {ares_data.get('street')}")
    print(f"  City: {ares_data.get('city')}")
    print(f"  ZIP: {ares_data.get('zip')}")
    print(f"  IČO: {ares_data.get('registration_no')}")
    print(f"  DIČ: {ares_data.get('vat_no')}")
else:
    print("\n✗ Company not found in ARES")

# Try another example
print(f"\n{'-'*60}")
test_ico2 = "27082440"  # Fakturoid s.r.o.
print(f"Looking up IČO: {test_ico2}")

ares_data2 = fakturoid.get_company_from_ares(test_ico2)
if ares_data2:
    print(f"\n✓ Company found in ARES:")
    print(f"  Name: {ares_data2.get('name')}")
    print(f"  Street: {ares_data2.get('street')}")
    print(f"  City: {ares_data2.get('city')}")


In [ ]:
# Test invoice submission using submit_expense method
from src.ai_extractor import InvoiceData

# Create sample invoice data (using real Czech company for ARES test)
sample_invoice = InvoiceData(
    invoice_number="TEST-001",
    issue_date="2024-10-08",
    supplier_name="Fakturoid s.r.o.",  # Real company - ARES will update details
    total_amount=1210.0,  # Including 21% VAT
    due_date="2024-10-22",
    supplier_address="Testovací 123, Praha",  # Will be replaced by ARES data
    supplier_ico="27082440",  # Real IČO - Fakturoid s.r.o.
    supplier_dic="CZ27082440",
    currency="CZK",
    variable_symbol="001",
    notes="Test invoice for API integration"
)

print("Sample invoice data:")
print(json.dumps(sample_invoice.model_dump(), indent=2, ensure_ascii=False))

# Submit to Fakturoid (automatically creates supplier if needed)
print("\n" + "="*60)
print("SUBMITTING TO FAKTUROID...")
print("="*60)

try:
    result = fakturoid.submit_expense(sample_invoice, auto_create_subject=True)
    print(f"\n✓ Expense created successfully!")
    print(f"Expense ID: {result.get('id')}")
    print(f"Number: {result.get('number')}")
    print(f"Supplier: {result.get('supplier_name')}")
    print(f"Total: {result.get('total')} {result.get('currency')}")
    print(f"\nView in Fakturoid: {result.get('html_url')}")
except Exception as e:
    print(f"\n✗ Failed to submit: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# Test with FOREIGN supplier (detailed address extraction)
from src.ai_extractor import InvoiceData

print("Testing foreign supplier with detailed address")
print("="*60)

# Example: German company
foreign_invoice = InvoiceData(
    invoice_number="DE-2025-001",
    issue_date="2025-01-15",
    supplier_name="Example GmbH",
    total_amount=500.00,  # EUR
    due_date="2025-02-15",
    # Detailed address fields
    supplier_street="Hauptstraße 123",
    supplier_city="Berlin",
    supplier_zip="10115",
    supplier_country="DE",
    # VAT number for foreign company
    supplier_vat_number="DE123456789",
    currency="EUR",
    variable_symbol="2025001",
    notes="Test invoice from German supplier"
)

print("\nForeign invoice data:")
print(json.dumps(foreign_invoice.model_dump(), indent=2, ensure_ascii=False))

print(f"\n{'='*60}")
print("Ready to submit - UNCOMMENT to create expense")
print(f"{'='*60}")

# UNCOMMENT to actually submit:
try:
    result = fakturoid.submit_expense(foreign_invoice, auto_create_subject=True)
    print(f"\n✓ Expense created successfully!")
    print(f"Expense ID: {result.get('id')}")
    print(f"Number: {result.get('number')}")
    print(f"Supplier: {result.get('supplier_name')}")
    print(f"Total: {result.get('total')} {result.get('currency')}")
    print(f"\nView in Fakturoid: {result.get('html_url')}")
except Exception as e:
    print(f"\n✗ Failed to submit: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# List recent expense invoices
try:
    invoices = fakturoid.list_expense_invoices(limit=10)
    print(f"Recent expense invoices ({len(invoices)}):")
    for inv in invoices:
        print(f"  - {inv.get('number')} | {inv.get('supplier_name')} | {inv.get('total')} {inv.get('currency')}")
except Exception as e:
    print(f"✗ Failed to list invoices: {e}")


In [ ]:
# Test with REAL invoice from PDF
from src.ai_extractor import AIExtractor
from src.document_processor import DocumentProcessor

print("Testing with real invoice from PDF...")
print("="*60)

# Get first invoice file
doc_processor = DocumentProcessor(config.directories.invoices)
files = doc_processor.list_invoice_files()

if files:
    test_file = files[0]
    print(f"\n📄 Processing: {test_file.name}")
    
    # Extract data using AI
    ai_extractor = AIExtractor(config)
    invoice_data_dict = ai_extractor.extract_invoice_data(test_file)
    
    # Convert to InvoiceData model
    invoice_data = InvoiceData(**invoice_data_dict)
    
    print(f"\n✓ Extracted data:")
    print(f"  Supplier: {invoice_data.supplier_name}")
    print(f"  Invoice #: {invoice_data.invoice_number}")
    print(f"  Date: {invoice_data.issue_date}")
    print(f"  Amount: {invoice_data.total_amount} {invoice_data.currency}")
    
    # Ask before submitting
    print(f"\n{'='*60}")
    print("Ready to submit to Fakturoid")
    print(f"{'='*60}")
    
    # UNCOMMENT to actually submit:
    try:
        result = fakturoid.submit_expense(invoice_data, auto_create_subject=True)
        print(f"\n✓ Expense created successfully!")
        print(f"Expense ID: {result.get('id')}")
        print(f"Number: {result.get('number')}")
        print(f"Supplier: {result.get('supplier_name')}")
        print(f"Total: {result.get('total')} {result.get('currency')}")
        print(f"\nView in Fakturoid: {result.get('html_url')}")
    except Exception as e:
        print(f"\n✗ Failed to submit: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No invoice files found in data/invoices/")
